In [ ]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    AutoTokenizer,
    AutoModelForCausalLM
)
from datasets import load_dataset
from torch.utils.data import Dataset
import torch
import os

# Step 0: Load dataset and tokenizer/model
csv_path = "C:/Users/gadep/OneDrive/Documents/wiki_hybrid_chunks_300.csv"  
dataset = load_dataset("csv", data_files={"train": csv_path})
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
SFT_OUTPUT_DIR = "./sft_tinyllama"
text_column = "chunk_text"  

# Step 1: Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Add pad token to GPT-2 and resize embeddings
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))

# Step 2: Prepare dataset
class SFTDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = item["input_ids"].clone()
        return item

# Step 3: Extract and preprocess training texts
texts = [item[text_column] for item in dataset["train"] if str(item[text_column]).strip() != ""]
sft_dataset = SFTDataset(texts, tokenizer)

# Step 4: Set training arguments
training_args = TrainingArguments(
    output_dir=SFT_OUTPUT_DIR,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=1,
    fp16=True,
    report_to="none",  # Disable external logging integrations
)

# Step 5: Initialize Trainer and start training
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sft_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

# Step 6: Save the fine-tuned model
trainer.save_model("./sft_finetuned_model")

C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\gadep\AppData\Local\Temp\ipykernel_8416\1263800855.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,4.096200
100,3.916100
150,3.953700
200,3.815900
250,3.785900


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gadep\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [7]:
print(dataset.column_names)


{'train': ['group_id', 'section', 'chunk_id', 'chunk_text']}


In [6]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map: 100%|██████████| 10022/10022 [00:09<00:00, 1106.68 examples/s]


In [4]:
from transformers import TrainingArguments

try:
    training_args = TrainingArguments(output_dir="./test_dir")
    print("TrainingArguments created with minimal args successfully")
except Exception as e:
    print("Error:", e)


TrainingArguments created with minimal args successfully


In [6]:
import inspect
from transformers import TrainingArguments

print(inspect.signature(TrainingArguments.__init__))


(self, output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: Optional[float] = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] = 'linear', lr_scheduler_kwargs: Union[dict, str, NoneType] = <factory>, warmup_ratio: float = 0.0, warmup_ste

In [10]:
print(tokenized_dataset.keys())


dict_keys(['train'])


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. Load raw dataset 
dataset = load_dataset("csv", data_files={"train": csv_path})

# Checking the columns of dataset before proceeding:
print(dataset["train"].column_names) 

# 2. Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# 3. Tokenize dataset

def tokenize_function(examples):
    return tokenizer(examples["chunk_text"], truncation=True, padding="max_length", max_length=128)

# Remove all original columns since tokenizer output replaces them
tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=dataset["train"].column_names
)

# 4. Split train dataset into train + validation
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1)
# split_dataset keys: 'train' and 'test' (use 'test' as validation)

# 5. Setup TrainingArguments
training_args = TrainingArguments(
    output_dir=SFT_OUTPUT_DIR,
    per_device_train_batch_size=1,  
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    num_train_epochs=3,
    fp16=False,  
    save_total_limit=2,
    logging_dir="./logs",
)

# 6. Data collator for causal LM (no masking)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 7. Trainer initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],  # validation split
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 8. Start training
trainer.train()

# 9. Save model and tokenizer
model.save_pretrained(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['group_id', 'section', 'chunk_id', 'chunk_text']


C:\Users\gadep\AppData\Local\Temp\ipykernel_12256\3887079148.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


In [1]:
import torch

if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU detected, training will use CPU and will be slow.")


No GPU detected, training will use CPU and will be slow.


In [3]:
from datasets import load_dataset


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

dataset = load_dataset("csv", data_files={"train": csv_path})
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["chunk_text"], truncation=True, max_length=128, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1)

training_args = TrainingArguments(
    output_dir=SFT_OUTPUT_DIR,
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=100,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_total_limit=2,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()
model.save_pretrained(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)


C:\Users\gadep\AppData\Local\Temp\ipykernel_16720\3047035269.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files={"train": csv_path})


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import LlamaTokenizer

try:
    tokenizer = LlamaTokenizer.from_pretrained(MODEL_NAME)
except Exception as e:
    print(f"Error loading tokenizer: {e}")


Error loading tokenizer: 
LlamaTokenizer requires the SentencePiece library but it was not found in your environment. Checkout the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.



In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:799: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.00 MB. The target location C:\Users\gadep\.cache\huggingface\hub\models--sshleifer--tiny-gpt2\blobs only has 0.00 MB free disk space.
  warnings.warn(
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gadep\.cache\huggingface\hub\models--sshleifer--tiny-gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environme

In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files={"train": csv_path})

batch_size = 8
for i in range(0, len(dataset), batch_size):
    batch = dataset.select(range(i, min(i+batch_size, len(dataset))))
    prompts = batch["chunk_text"] 


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import PPOTrainer, PPOConfig
import torch
from datasets import load_dataset

ppo_config = PPOConfig(learning_rate=1e-5, batch_size=1, mini_batch_size=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

reward_model = None
value_model = None

# Load a dataset that has a 'chunk_text' field or similar text prompts
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# I n the dataset, map the relevant column:
dataset = dataset.rename_column("text", "chunk_text")

ppo_trainer = PPOTrainer(
    ppo_config,
    None,
    model,
    ref_model=ref_model,
    reward_model=reward_model,
    train_dataset=dataset,
    value_model=value_model,
    tokenizer=tokenizer,
)

def reward_fn(response: str) -> float:
    reward = 0.0
    if "summary" in response.lower():
        reward += 1.0
    if len(response.split()) < 80:
        reward += 0.5
    return reward

batch_size = 4
for i in range(0, len(dataset), batch_size):
    batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
    prompts = batch["chunk_text"]

    tokenized_prompts = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).input_ids.to(device)

    responses = []
    for input_ids in tokenized_prompts:
        input_ids = input_ids.unsqueeze(0)
        output_ids = model.generate(input_ids, max_new_tokens=50)
        decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        responses.append(decoded)

    rewards = [reward_fn(resp) for resp in responses]

    ppo_trainer.step(prompts, responses, rewards)

    print(f"Processed batch {i // batch_size + 1}")
    for prompt, resp, reward in zip(prompts, responses, rewards):
        print(f"Prompt: {prompt[:60]}... → Reward: {reward:.2f}")
        print(f"Response: {resp[:60]}...\n")

model.save_pretrained("ppo_tinyllama")
tokenizer.save_pretrained("ppo_tinyllama")


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gadep\.cache\huggingface\hub\datasets--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to

TypeError: PPOTrainer.__init__() got an unexpected keyword argument 'tokenizer'

In [4]:
from trl import PPOTrainer
import inspect
print(inspect.signature(PPOTrainer.__init__))


(self, args: trl.trainer.ppo_config.PPOConfig, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.image_processing_utils.BaseImageProcessor, transformers.feature_extraction_utils.FeatureExtractionMixin, transformers.processing_utils.ProcessorMixin, NoneType], model: torch.nn.modules.module.Module, ref_model: Optional[torch.nn.modules.module.Module], reward_model: torch.nn.modules.module.Module, train_dataset: datasets.arrow_dataset.Dataset, value_model: torch.nn.modules.module.Module, data_collator: Optional[transformers.data.data_collator.DataCollatorWithPadding] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, dict[str, datasets.arrow_dataset.Dataset], NoneType] = None, optimizers: tuple[torch.optim.optimizer.Optimizer, torch.optim.lr_scheduler.LambdaLR] = (None, None), callbacks: Optional[list[transformers.trainer_callback.TrainerCallback]] = None, peft_config: Optional[ForwardRef('PeftConfig')] = None) -> None
